# Proyecto Final Parte 4 - Generación Musical y Reproducción con Modelos del Lenguaje (MidiTok REMI)

### Nombre: Roberto Alejandro Chagra Martinez
### Código: 208447128

---

## 🎵 Modelado Generativo Simbólico con Tokenización REMI y Redes Neuronales

Este cuaderno constituye la **Parte 4 del Proyecto Final**, integrándose directamente con los entregables previos (Parte 1: Visualización, Parte 2: Reproducción y Parte 3: Tokenización).

En esta entrega avanzamos desde la tokenización y preparación de datos hacia la **generación real de nuevas piezas musicales**:
1. Carga del corpus de archivos MIDI pre-procesados y tokenizados con **REMI de MidiTok**.
2. Definición del modelo de red neuronal autorregresivo con capa de **Embedding musical**.
3. Entrenamiento supervisado para predecir el siguiente evento musical con `CrossEntropyLoss`.
4. Interfaz interactiva de generación (*ipywidgets*) para seleccionar temperatura, creatividad y tamaño de la composición.
5. Decodificación instantánea a archivo `.mid` con `tokenizer.decode()`.
6. Visualización de la pieza generada en **Piano Roll** y **Reproductor de Audio integrado**.


In [ ]:
#@title Preparación: Instalación e Importación de Bibliotecas
# !pip install -q miditok symusic pretty_midi torch numpy pandas matplotlib seaborn ipywidgets soundfile

import sys
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass

import os
import random
import math
import wave
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import symusic
import miditok
from miditok import REMI, TokenizerConfig
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
try:
    from IPython.display import display, Audio, HTML
except ImportError:
    class Audio:
        def __init__(self, filename=None, *args, **kwargs): self.filename = filename
        def __repr__(self): return f"<Audio: {self.filename}>"
    class HTML:
        def __init__(self, data): self.data = data
        def __repr__(self): return self.data
    display = print

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

# Semilla fija
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("✅ Entorno listo para la Generación Musical con MidiTok y PyTorch")


In [ ]:
#@title 1. Carga del Dataset Limpio y Tokenizador REMI
RUTA_DATASET_CLEAN = Path("dataset_midi_clean")
RUTA_RESULTADOS = Path("results")
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

# Configuración del Tokenizador REMI
config_remi = TokenizerConfig(
    pitch_range=(21, 109),
    beat_res={(0, 4): 8, (4, 12): 4},
    num_velocities=32,
    use_chords=True,
    use_rests=True,
    use_tempos=True,
    use_time_signatures=True,
    use_programs=False,
)
tokenizer = REMI(config_remi)
vocab_size = len(tokenizer)
pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

archivos_limpios = sorted(list(RUTA_DATASET_CLEAN.glob("*.mid")))
print(f"Archivos limpios encontrados: {len(archivos_limpios)}")
print(f"Tamaño del vocabulario REMI: {vocab_size} tokens únicos")

# Tokenizar y fragmentar
SECUENCIA_MAX = 256
STRIDE = 128
muestras_entrenamiento = []

for p in archivos_limpios:
    seq = tokenizer(p)
    ids = seq[0].ids if isinstance(seq, list) else seq.ids
    if len(ids) <= SECUENCIA_MAX:
        muestras_entrenamiento.append(ids)
    else:
        for s_idx in range(0, len(ids) - SECUENCIA_MAX + 1, STRIDE):
            muestras_entrenamiento.append(ids[s_idx : s_idx + SECUENCIA_MAX])

print(f"Total de secuencias de entrenamiento generadas: {len(muestras_entrenamiento)}")


In [ ]:
#@title 2. Definición del Dataset y Modelo de Red Neuronal para Tokens
class DatasetMusicalAutorregresivo(Dataset):
    def __init__(self, secuencias, max_len=256, pad_id=0):
        self.x = []
        self.y = []
        for s in secuencias:
            if len(s) < 4: continue
            cur = s[:max_len]
            x_seq = cur[:-1]
            y_seq = cur[1:]
            pad_len = (max_len - 1) - len(x_seq)
            self.x.append(torch.tensor(x_seq + [pad_id]*pad_len, dtype=torch.long))
            self.y.append(torch.tensor(y_seq + [pad_id]*pad_len, dtype=torch.long))
            
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i]

dataset = DatasetMusicalAutorregresivo(muestras_entrenamiento, max_len=SECUENCIA_MAX, pad_id=pad_id)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

class GeneradorMusicalTokens(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.2, pad_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden=None):
        emb = self.embedding(x)
        out, hidden = self.lstm(emb, hidden)
        logits = self.fc(out)
        return logits, hidden

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo = GeneradorMusicalTokens(vocab_size, embed_dim=128, hidden_dim=256, num_layers=2, pad_id=pad_id).to(device)
criterio = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizador = optim.Adam(modelo.parameters(), lr=0.003)

print("Arquitectura de la Red:")
print(modelo)


In [ ]:
#@title 3. Entrenamiento del Modelo Generativo
EPOCAS = 30
modelo.train()
print(f"Iniciando entrenamiento por {EPOCAS} épocas...")

for ep in range(1, EPOCAS + 1):
    acum_loss = 0.0
    batches = 0
    for bx, by in dataloader:
        bx, by = bx.to(device), by.to(device)
        optimizador.zero_grad()
        logits, _ = modelo(bx)
        loss = criterio(logits.view(-1, vocab_size), by.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        optimizador.step()
        acum_loss += loss.item()
        batches += 1
    if ep % 5 == 0 or ep == 1:
        print(f"Época [{ep:02d}/{EPOCAS}] - Pérdida promedio: {acum_loss / max(1, batches):.4f}")

print("\n¡Entrenamiento completado!")


In [ ]:
#@title 4. Función de Generación Autoregresiva
def generar_musica(modelo, semilla, pasos=300, temperatura=0.85, top_k=15, max_context=255):
    modelo.eval()
    secuencia = list(semilla)
    with torch.no_grad():
        for _ in range(pasos):
            ctx = secuencia[-max_context:]
            x_in = torch.tensor([ctx], dtype=torch.long, device=device)
            logits, _ = modelo(x_in)
            last_logits = logits[0, -1, :] / max(temperatura, 1e-4)
            if top_k > 0:
                vals, idxs = torch.topk(last_logits, min(top_k, last_logits.size(-1)))
                filtro = torch.full_like(last_logits, -float("Inf"))
                filtro.scatter_(0, idxs, vals)
                last_logits = filtro
            probs = torch.softmax(last_logits, dim=-1)
            sig_token = torch.multinomial(probs, 1).item()
            secuencia.append(sig_token)
    return secuencia


In [ ]:
#@title 5. Interfaz Interactiva de Generación (Widgets)
if widgets is not None:
    slider_pasos = widgets.IntSlider(value=300, min=100, max=800, step=50, description='Tokens:')
    slider_temp = widgets.FloatSlider(value=0.85, min=0.2, max=1.5, step=0.05, description='Temperatura:')
    slider_topk = widgets.IntSlider(value=15, min=5, max=50, step=5, description='Top-K:')
    boton_generar = widgets.Button(description="Generar Música", button_style='primary')
    salida_generacion = widgets.Output()

    display(widgets.VBox([slider_pasos, slider_temp, slider_topk, boton_generar, salida_generacion]))

    def on_click_generar(b):
        with salida_generacion:
            salida_generacion.clear_output()
            print("Iniciando generación con la IA...")
            semilla = muestras_entrenamiento[0][:20]
            gen_tokens = generar_musica(
                modelo,
                semilla,
                pasos=slider_pasos.value,
                temperatura=slider_temp.value,
                top_k=slider_topk.value
            )
            score_nuevo = tokenizer.decode([gen_tokens])
            dest_mid = RUTA_RESULTADOS / "composicion_ia_parte4.mid"
            score_nuevo.dump_midi(dest_mid)
            print(f"✅ Archivo MIDI guardado en: {dest_mid}")
            print(f" - Notas generadas: {score_nuevo.note_num()}")
            print(f" - Duración estimada: {float(score_nuevo.to('second').end()):.2f}s")
            
    boton_generar.on_click(on_click_generar)
else:
    print("[INFO] Widgets interactivos disponibles al ejecutar dentro de Jupyter Notebook / Colab.")


In [ ]:
#@title 6. Visualización Piano Roll y Reproductor de Audio
def visualizar_y_reproducir(ruta_midi):
    score = symusic.Score.from_file(ruta_midi)
    
    # 1. Piano Roll
    plt.figure(figsize=(12, 4))
    for t in score.tracks:
        if not t.notes: continue
        st = [n.time / float(score.ticks_per_quarter) for n in t.notes]
        dur = [n.duration / float(score.ticks_per_quarter) for n in t.notes]
        pitches = [n.pitch for n in t.notes]
        plt.barh(pitches, dur, left=st, height=0.8, color="#3498db", alpha=0.8)
    plt.title(f"Piano Roll: {Path(ruta_midi).name}", fontsize=12, fontweight='bold')
    plt.xlabel("Pulsos (Beats)")
    plt.ylabel("Tono MIDI")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()
    
    # 2. Síntesis a WAV
    dur_sec = float(score.to("second").end()) + 1.0
    sr = 22050
    buf = np.zeros(int(dur_sec * sr), dtype=np.float32)
    s_sec = score.to("second")
    
    for tr in s_sec.tracks:
        for n in tr.notes:
            freq = 440.0 * (2.0 ** ((n.pitch - 69) / 12.0))
            t_start = float(n.time)
            t_dur = max(0.05, float(n.duration))
            vel = n.velocity / 127.0
            s_idx = int(t_start * sr)
            e_idx = min(len(buf), s_idx + int(t_dur * sr))
            n_s = e_idx - s_idx
            if n_s > 0:
                t_vec = np.linspace(0, t_dur, n_s, endpoint=False)
                decay = np.exp(-3.0 * t_vec / t_dur)
                sig = vel * decay * (0.7 * np.sin(2*np.pi*freq*t_vec) + 0.3 * np.sin(4*np.pi*freq*t_vec))
                buf[s_idx:e_idx] += sig
                
    max_amp = np.max(np.abs(buf))
    if max_amp > 0: buf = (buf / max_amp) * 0.9
    wav_path = Path(ruta_midi).with_suffix(".wav")
    with wave.open(str(wav_path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes((buf * 32767).astype(np.int16).tobytes())
        
    print(f"🔊 Reproductor de Audio ({wav_path.name}):")
    display(Audio(str(wav_path)))

# Probar con el archivo generado
midi_demo = RUTA_RESULTADOS / "mi_primera_composicion_ia_remi.mid"
if not midi_demo.exists():
    # Generar uno de prueba si aún no se pulsó el botón
    sem = muestras_entrenamiento[0][:20]
    g = generar_musica(modelo, sem, pasos=250)
    sc = tokenizer.decode([g])
    sc.dump_midi(midi_demo)

visualizar_y_reproducir(str(midi_demo))
